# NYC Collision Analysis: Datalake & Athena Setup
This notebook sets up the foundational infrastructure on AWS. We're moving our local data into S3 and registering it with Athena so we can run SQL queries on the full dataset.

In [ ]:
!pip install "sagemaker<3.0.0" awswrangler

import boto3
import sagemaker
import pandas as pd
import awswrangler as wr
import os

# 1. Connections
sess = sagemaker.Session()
bucket = sess.default_bucket()
region = sess.boto_region_name
prefix = "aai-540-group6/nyc-collisions"

s3_path = f"s3://{bucket}/{prefix}/data/raw/"
print(f"Target Datalake location: {s3_path}")

### Load the Merged Sample
We use the 1,000-row sample created in Phase 1 as our starting point.

In [ ]:
csv_file = '../data/raw/merged_sample.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"Found sample data: {df.shape[0]} rows merged.")
else:
    print("Error: CSV missing. Run src/data_loader.py first.")

### Populate the Datalake
Push to S3 and sync the Glue Catalog (Athena) in one go.

In [ ]:
db = 'nyc_collision_db'
table = 'merged_raw'

# Make sure DB exists
wr.catalog.create_database(db, exist_ok=True)

# Push data
wr.s3.to_csv(
    df=df,
    path=s3_path,
    dataset=True,
    database=db,
    table=table,
    index=False,
    mode='overwrite'
)

print(f"Database '{db}' updated. Table '{table}' ready for SQL.")

### Quick SQL Check
Run some Athena queries to verify the ingestion.

In [ ]:
# Check distributions across boroughs
q1 = f"SELECT borough, count(*) as total FROM {db}.{table} GROUP BY borough ORDER BY total DESC"
res1 = wr.athena.read_sql_query(q1, database=db)
display(res1)

In [ ]:
# Look at vehicle types involved
q2 = f"SELECT vehicle_type_code1, count(*) as freq FROM {db}.{table} WHERE vehicle_type_code1 IS NOT NULL GROUP BY vehicle_type_code1 ORDER BY freq DESC LIMIT 5"
res2 = wr.athena.read_sql_query(q2, database=db)
display(res2)